# Creativity metrics for the EGRA story methods

Reads the story-review workbook (one sheet per method, every story with its
coherence verdict) from your Google Drive and scores each method's
**coherent** stories on four measures, each against the untouched baseline:

1. **Vendi score** - the standard embedding Vendi: Qwen3-Embedding-0.6B over the
   first 40 words, isolated stories trimmed, averaged over random subsamples of
   one common size, with 95% intervals on each method's difference from the
   baseline and from nucleus sampling. Higher is more varied.
2. **NoveltyBench distinct-10** (Zhang et al., 2025) - how many *genuinely
   different* stories there are among 10, decided by a DeBERTa-v3-large judge
   fine-tuned on human labels; rewording the same story does not count as new.
   Also the share of all pairs judged the same story. Higher distinct / lower
   same-story share is more varied.
3. **Opening subjects** - the grammatical subject of each story's first
   sentence (names grouped as "a named character", pronouns as "a pronoun"):
   how many different ones, the effective number (evenness-weighted), and the
   share taken by the single most common one.
4. **Homogeneity (Shaib et al., 2025)** - over the first 100 words: how well the
   set compresses (higher = more repetitive), the same over part-of-speech
   sequences (repetitive sentence structure), and the share of each story's
   four-word phrases that also appear in another story.

**To run:** upload the workbook to your Drive, put its path in the settings cell
below, set Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.
Results are printed as a table and saved next to the workbook as
`creativity_metrics.csv` and `creativity_metrics.json`.

In [ ]:
# @title Settings
EXCEL_PATH = "/content/drive/MyDrive/story_review_ten_methods.xlsx"  # @param {type:"string"}
BASELINE_SHEET = "01 Baseline"  # @param {type:"string"}
NUCLEUS_SHEET = "02 Nucleus"  # @param {type:"string"}
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"  # @param {type:"string"}

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install sentencepiece protobuf sentence-transformers openpyxl
    !python -m spacy download en_core_web_sm -q
assert os.path.exists(EXCEL_PATH), f"no workbook at {EXCEL_PATH} -- check the path in the settings cell"
OUT_DIR = os.path.dirname(EXCEL_PATH)
print("workbook:", EXCEL_PATH)

In [ ]:
# Read every method sheet: the table under the "story #" header, coherent stories only.
from openpyxl import load_workbook

wb = load_workbook(EXCEL_PATH, read_only=True, data_only=True)
STORIES = {}
for ws in wb.worksheets:
    if ws.title.lower() == "summary":
        continue
    rows = list(ws.iter_rows(values_only=True))
    head = next((i for i, r in enumerate(rows) if r and r[0] == "story #"), None)
    if head is None:
        print(f"skipping sheet {ws.title!r}: no story table")
        continue
    cols = {str(c).strip().lower(): j for j, c in enumerate(rows[head]) if c is not None}
    keep = [r[cols["story"]] for r in rows[head + 1:]
            if r and r[cols["coherent"]] and str(r[cols["coherent"]]).strip().lower() == "yes"
            and r[cols["story"]]]
    STORIES[ws.title] = [str(t) for t in keep]
for name, texts in STORIES.items():
    print(f"  {name:28s} {len(texts)} coherent stories")
assert BASELINE_SHEET in STORIES and NUCLEUS_SHEET in STORIES, "check the baseline and nucleus sheet names"

In [ ]:
import numpy as np, random, math, json, gzip, time
import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)


def first_words(t, n):
    return " ".join(t.split()[:n])


def vendi(e):
    e = e / np.linalg.norm(e, axis=1, keepdims=True)
    w = np.linalg.eigvalsh(e @ e.T / len(e))
    w = w[w > 1e-12]
    return float(np.exp(-(w * np.log(w)).sum()))


def trim_isolated(e, z=3.5):
    """Drop stories far from the rest of their own set: robust z of mean similarity, one-sided."""
    en = e / np.linalg.norm(e, axis=1, keepdims=True)
    s = en @ en.T
    np.fill_diagonal(s, np.nan)
    m = np.nanmean(s, axis=1)
    med = np.median(m)
    mad = np.median(np.abs(m - med)) * 1.4826 or 1e-9
    return e[(m - med) / mad > -z]


def interval(a):
    return [float(np.mean(a)), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

In [ ]:
# 1. Vendi score
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True, device=DEVICE)
EMB = {n: trim_isolated(embedder.encode([first_words(t, 40) for t in texts],
                                        convert_to_numpy=True, normalize_embeddings=True))
       for n, texts in STORIES.items()}
size = min(len(e) for e in EMB.values()) // 2
rng = np.random.default_rng(0)
DRAWS = {n: np.array([vendi(e[rng.choice(len(e), size, replace=False)]) for _ in range(400)])
         for n, e in EMB.items()}
VENDI = {}
for n, d in DRAWS.items():
    VENDI[n] = dict(vendi=float(d.mean()),
                    vs_baseline=interval(d - DRAWS[BASELINE_SHEET][rng.permutation(400)]),
                    vs_nucleus=interval(d - DRAWS[NUCLEUS_SHEET][rng.permutation(400)]))
print(f"Vendi over subsamples of {size} stories")
for n, v in VENDI.items():
    print(f"  {n:28s} {v['vendi']:.2f}  vs baseline {v['vs_baseline'][0]:+.2f}  vs nucleus {v['vs_nucleus'][0]:+.2f}")
del embedder
torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
# 2. NoveltyBench distinct-10
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
nb_judge = AutoModelForSequenceClassification.from_pretrained(
    "yimingzhang/deberta-v3-large-generation-similarity").to(DEVICE).eval()
THRESHOLD, K, SUBSETS = 0.102, 10, 500
_enc = {}


def nb_enc(s):
    if s not in _enc:
        _enc[s] = nb_tok.encode(s, truncation=True, max_length=128, add_special_tokens=False)
    return _enc[s]


def nb_scores(pairs, batch=64):
    out = []
    for i in range(0, len(pairs), batch):
        ids, tts = [], []
        for a, b in pairs[i:i + batch]:
            x = [nb_tok.cls_token_id] + nb_enc(a) + [nb_tok.sep_token_id]
            first = len(x)
            x += nb_enc(b) + [nb_tok.sep_token_id]
            ids.append(x)
            tts.append([0] * first + [1] * (len(x) - first))
        L = max(len(x) for x in ids)
        att = [[1] * len(x) + [0] * (L - len(x)) for x in ids]
        ids = [x + [nb_tok.pad_token_id] * (L - len(x)) for x in ids]
        tts = [t + [0] * (L - len(t)) for t in tts]
        with torch.inference_mode():
            lg = nb_judge(input_ids=torch.tensor(ids, device=DEVICE),
                          token_type_ids=torch.tensor(tts, device=DEVICE),
                          attention_mask=torch.tensor(att, device=DEVICE)).logits
        out += lg.float().softmax(-1)[:, 1].cpu().tolist()
    return out


def distinct_k(texts, k=K, subsets=SUBSETS, seed=0):
    n = len(texts)
    k = min(k, n)
    idx = [(i, j) for i in range(n) for j in range(i)]
    sc = nb_scores([(texts[i], texts[j]) for i, j in idx])
    same = {}
    for (i, j), s in zip(idx, sc):
        same[(i, j)] = same[(j, i)] = s > THRESHOLD
    r = random.Random(seed)
    counts = []
    for _ in range(subsets):
        heads = []
        for i in r.sample(range(n), k):
            if not any(same[(i, h)] for h in heads):   # joins the first class it matches
                heads.append(i)
        counts.append(len(heads))
    counts.sort()
    return dict(distinct=sum(counts) / len(counts), low=counts[int(0.025 * subsets)],
                high=counts[int(0.975 * subsets) - 1],
                same_story_pairs=sum(s > THRESHOLD for s in sc) / len(sc))


NOVELTY = {}
for n, texts in STORIES.items():
    t0 = time.time()
    NOVELTY[n] = distinct_k(texts)
    v = NOVELTY[n]
    print(f"  {n:28s} distinct-10 {v['distinct']:.2f} ({v['low']}-{v['high']})  "
          f"same-story pairs {v['same_story_pairs']:.1%}  {time.time() - t0:.0f}s")
del nb_judge
torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
# 3. Opening subjects and 4. homogeneity
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")


def opening_subject(text):
    sent = next(nlp(first_words(text, 60)).sents)
    for t in sent:
        if t.dep_ in ("nsubj", "nsubjpass") and t.head.dep_ == "ROOT":
            if t.pos_ == "PROPN":
                return "a named character"
            if t.pos_ == "PRON":
                return "a pronoun"
            return t.lemma_.lower()
    chunks = list(sent.noun_chunks)
    if chunks:
        root = chunks[0].root
        return "a named character" if root.pos_ == "PROPN" else root.lemma_.lower()
    return "(no subject)"


k_common = min(len(t) for t in STORIES.values())
OPEN, HOMOG = {}, {}
for n, texts in STORIES.items():
    subs = [opening_subject(t) for t in texts]
    pos = [" ".join(tok.pos_ for tok in nlp(first_words(t, 100))) for t in texts]
    r = random.Random(0)
    uniq, top, eff, cr, pcr, rep = [], [], [], [], [], []
    for _ in range(200):
        pick = r.sample(range(len(texts)), k_common)
        c = Counter(subs[i] for i in pick)
        uniq.append(len(c))
        top.append(max(c.values()) / k_common)
        eff.append(math.exp(-sum(v / k_common * math.log(v / k_common) for v in c.values())))
        raw = "\n".join(first_words(texts[i], 100) for i in pick).encode()
        cr.append(len(raw) / len(gzip.compress(raw)))
        praw = "\n".join(pos[i] for i in pick).encode()
        pcr.append(len(praw) / len(gzip.compress(praw)))
        grams = [set(zip(*(first_words(texts[i], 100).lower().split()[j:] for j in range(4)))) for i in pick]
        shared = [len(g & set().union(*(grams[j] for j in range(len(grams)) if j != a))) / max(len(g), 1)
                  for a, g in enumerate(grams)]
        rep.append(sum(shared) / len(shared))
    OPEN[n] = dict(unique=float(np.mean(uniq)), effective=float(np.mean(eff)), top_share=float(np.mean(top)),
                   most_common=Counter(subs).most_common(5))
    HOMOG[n] = dict(compression=float(np.mean(cr)), pos_compression=float(np.mean(pcr)),
                    shared_4grams=float(np.mean(rep)))
print(f"opening subjects and homogeneity at {k_common} coherent stories per method")

In [ ]:
# The table
import pandas as pd

rows = []
for n in STORIES:
    v, nv, o, h = VENDI[n], NOVELTY[n], OPEN[n], HOMOG[n]
    ci = lambda x: f"{x[0]:+.2f} [{x[1]:+.2f}, {x[2]:+.2f}]"
    rows.append({
        "method": n, "coherent stories": len(STORIES[n]),
        "Vendi": round(v["vendi"], 2),
        "Vendi vs baseline": "baseline" if n == BASELINE_SHEET else ci(v["vs_baseline"]),
        "Vendi vs nucleus": "reference" if n == NUCLEUS_SHEET else ci(v["vs_nucleus"]),
        "distinct-10": f"{nv['distinct']:.2f} ({nv['low']}-{nv['high']})",
        "same-story pairs": f"{nv['same_story_pairs']:.1%}",
        "opening subjects (unique)": round(o["unique"], 1),
        "opening subjects (effective)": round(o["effective"], 1),
        "top opening share": f"{o['top_share']:.0%}",
        "most common openings": ", ".join(f"{w} {c}" for w, c in o["most_common"][:3]),
        "compression": round(h["compression"], 2),
        "POS compression": round(h["pos_compression"], 2),
        "shared 4-grams": f"{h['shared_4grams']:.1%}",
    })
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT_DIR, "creativity_metrics.csv"), index=False)
json.dump(dict(vendi=VENDI, vendi_subsample=size, novelty=NOVELTY, openings=OPEN, homogeneity=HOMOG,
               opening_and_homogeneity_stories=k_common),
          open(os.path.join(OUT_DIR, "creativity_metrics.json"), "w"), indent=1, default=str)
print("saved to", OUT_DIR)
pd.set_option("display.max_columns", None, "display.width", 250)
table